# 04. Nested Models, Discriminated Unions & JSON Schema

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuichi110/drawlib-examples/blob/main/docs/pydantic/en/04_nested_unions_and_schema.ipynb)

Real-world REST APIs, webhook event streams, and LLM tool/structured-output contracts rarely validate flat records in isolation. Instead, production systems compose **nested `BaseModel` hierarchies**, route polymorphic payloads in `O(1)` time via **tagged Discriminated Unions (`Field(discriminator=...)` + `TypeAdapter`)**, and export a single canonical **JSON Schema (`model_json_schema()`)** that simultaneously powers FastAPI's OpenAPI 3.1 documentation and grammar-constrained LLM decoding (such as Google GenAI SDK's `response_schema`).

![Figure 4-1: O(1) Routing with Tagged Discriminated Unions and TypeAdapter](https://examples.drawlib.com/pydantic/images/04_nested_unions_and_schema/ch04_discriminated_union.png)

*Figure 4-1: O(1) Routing with Tagged Discriminated Unions and TypeAdapter*

---

## 1. Nested Models & Discriminated Unions (`Field(discriminator=...)` + `TypeAdapter`)

With an untagged Python union (`CardPayment | BankTransfer | WalletPayment`), a validator must try each branch sequentially from left to right—wasting CPU cycles on misses and emitting noisy multi-branch `ValidationError` reports whenever an input fails. Attaching a `Literal[...]` tag field (such as `kind`) to every variant and combining them via `Annotated[..., Field(discriminator="kind")]` turns union dispatch into an **`O(1)` hash-table lookup** inside `pydantic-core`:

- **Nested `BaseModel` Composition**: Embed `PaymentMethod` directly inside `CheckoutOrder` so nested dictionaries are validated and instantiated into concrete subclass instances (`CardPayment`, `BankTransfer`, or `WalletPayment`) in a single pass.
- **Standalone Collection Validation (`TypeAdapter`)**: When validating bare lists (`list[PaymentMethod]`) or webhook arrays without wrapping them in an outer `BaseModel`, compile a reusable `TypeAdapter(list[PaymentMethod])` once at module import time.
- **Pinpoint Error Diagnostics**: If `{"kind": "card", "last4": "12AB"}` fails regex validation, Pydantic jumps straight to `CardPayment` and reports **only** the exact `'card' -> 'last4'` error rather than complaining about missing `bank_code` or `provider` fields.

In [ ]:
!pip install -q pydantic

In [ ]:
from typing import Annotated, Literal

from pydantic import BaseModel, Field, TypeAdapter, ValidationError


class CardPayment(BaseModel):
    kind: Literal["card"] = "card"
    last4: str = Field(pattern=r"^\d{4}$", description="Masked last 4 digits of the card.")
    amount: int = Field(gt=0, description="Payment amount in cents.")


class BankTransfer(BaseModel):
    kind: Literal["bank"] = "bank"
    bank_code: str = Field(min_length=4, max_length=11, description="SWIFT or bank routing code.")
    amount: int = Field(gt=0, description="Transfer amount in cents.")


class WalletPayment(BaseModel):
    kind: Literal["wallet"] = "wallet"
    provider: Literal["apple_pay", "google_pay"] = Field(description="Digital wallet provider.")
    amount: int = Field(gt=0, description="Payment amount in cents.")


# O(1) Tagged Discriminated Union routed by the 'kind' literal field
PaymentMethod = Annotated[
    CardPayment | BankTransfer | WalletPayment,
    Field(discriminator="kind"),
]


class CheckoutOrder(BaseModel):
    order_id: str = Field(description="Unique order identifier.")
    customer_email: str = Field(description="Buyer confirmation email address.")
    payment: PaymentMethod = Field(description="Polymorphic payment method payload.")


# 1. Validate a nested CheckoutOrder payload
order = CheckoutOrder.model_validate(
    {
        "order_id": "ORD-2026-0042",
        "customer_email": "alice@example.com",
        "payment": {"kind": "card", "last4": "4242", "amount": 4800},
    }
)
print(f"Nested order {order.order_id} -> {type(order.payment).__name__}: {order.payment}")

# 2. Validate a bare batch list[PaymentMethod] directly via TypeAdapter
payment_batch_adapter = TypeAdapter(list[PaymentMethod])
payments = payment_batch_adapter.validate_python(
    [
        {"kind": "card", "last4": "4242", "amount": 4800},
        {"kind": "bank", "bank_code": "BOFAUS3N", "amount": 12500},
        {"kind": "wallet", "provider": "google_pay", "amount": 1900},
    ]
)
print("Batch routed types:", [type(item).__name__ for item in payments])

# 3. Pinpoint O(1) error reporting when a specific variant field is invalid
try:
    payment_batch_adapter.validate_python(
        [{"kind": "card", "last4": "12AB", "amount": 4800}]
    )
except ValidationError as exc:
    err = exc.errors(include_url=False)[0]
    print(f"Pinpoint error ({exc.error_count()} total): loc={err['loc']} -> {err['msg']}")

---

## 2. Generating Standard JSON Schema (`model_json_schema()`) for APIs & LLMs

Beyond runtime validation, every Pydantic v2 `BaseModel` (and `TypeAdapter`) compiles directly into **JSON Schema Draft 2020-12** via `.model_json_schema()`. Defining constraints, `Literal` taxonomies, `Field(discriminator=...)` mappings, and `Field(description=...)` hints once in Python establishes a **Single Source of Truth** across your entire engineering stack:

![Figure 4-2: One Pydantic Model Powering FastAPI OpenAPI Docs and LLM Structured Outputs](https://examples.drawlib.com/pydantic/images/04_nested_unions_and_schema/ch04_json_schema_ecosystem.png)

*Figure 4-2: One Pydantic Model Powering FastAPI OpenAPI Docs and LLM Structured Outputs*

Calling `CheckoutOrder.model_json_schema()` emits reusable sub-model definitions under `$defs`, an explicit `discriminator` mapping (`"card"` -> `"#/$defs/CardPayment"`), `required` property arrays, regex `pattern` / numeric bounds, and `description` strings. Notice how cleanly this single dictionary bridges backend APIs and AI engineering—in fact, passing `response_schema=CheckoutOrder` in the **Google GenAI SDK Cookbook** uses this exact schema introspection under the hood to constrain Gemini's token decoding to valid `CheckoutOrder` JSON:

In [ ]:
import json

schema = CheckoutOrder.model_json_schema()

print("Top-level schema keys:", list(schema.keys()))
print("Nested $defs models:", list(schema["$defs"].keys()))
print("\nFull JSON Schema (Draft 2020-12):")
print(json.dumps(schema, indent=2))